In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree(
    '/content/drive/MyDrive/ai_chat_bot',
    '/content/ai_chat_bot'
)
print("Folder Loaded!")

In [ ]:
import torch
from transformers import BlenderbotTokenizer, BlenderbotForConditionalGeneration

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

tokenizer = BlenderbotTokenizer.from_pretrained('/content/ai_chat_bot/MentalHealth')
model = BlenderbotForConditionalGeneration.from_pretrained('/content/ai_chat_bot/MentalHealth')
model = model.to(device)
model.eval()
print("Model Loaded!")
print("Tokenizer Loaded!")

In [ ]:
import tensorflow as tf
import pickle
import numpy as np
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2

# architecture
def create_bilstm_model(vocab_size, max_length, embedding_dim=128,
                         lstm_units=128, num_classes=28, dropout_rate=0.3):
    inputs = Input(shape=(max_length,))
    x = Embedding(vocab_size, embedding_dim,
                  input_length=max_length, mask_zero=True)(inputs)
    x = Dropout(0.2)(x)
    x = Bidirectional(LSTM(lstm_units, return_sequences=True,
                           dropout=0.3, recurrent_dropout=0.2))(x)
    x = Bidirectional(LSTM(64, return_sequences=False,
                           dropout=0.3, recurrent_dropout=0.2))(x)
    x = Dense(256, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(dropout_rate)(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(0.001))(x)
    x = Dropout(dropout_rate)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs, outputs)
    return model

# Tokenizer load
with open('/content/ai_chat_bot/BiLSTM/goemotion_tokenizer.pkl', 'rb') as f:
    bilstm_tokenizer = pickle.load(f)

with open('/content/ai_chat_bot/BiLSTM/goemotion_label_mapping.pkl', 'rb') as f:
    emotion_labels_map = pickle.load(f)

with open('/content/ai_chat_bot/BiLSTM/goemotion_max_length.pkl', 'rb') as f:
    MAX_LENGTH = pickle.load(f)

vocab_size = min(len(bilstm_tokenizer.word_index) + 1, 20000)
print(f"Vocab Size: {vocab_size}")
print(f"Max Length: {MAX_LENGTH}")

# Model
bilstm_model = create_bilstm_model(vocab_size, MAX_LENGTH)

# Weights load
bilstm_model.load_weights('/content/ai_chat_bot/BiLSTM/bilstm_goemotion_balanced.h5')
print("BiLSTM Model Loaded!")

In [ ]:
import numpy as np
import tensorflow as tf
import torch
from transformers import BlenderbotTokenizer, BlenderbotForConditionalGeneration

# Emotion Detection
def detect_emotion(text):
    tokens = bilstm_tokenizer.texts_to_sequences([text])
    padded = tf.keras.preprocessing.sequence.pad_sequences(tokens, maxlen=MAX_LENGTH, padding='post')
    predictions = bilstm_model.predict(padded, verbose=0)[0]
    top_idx = np.argmax(predictions)
    return emotion_labels_map[top_idx]

# Response Generation
def generate_response(user_input, emotion):
    prefixed_input = f"I feel {emotion}. {user_input}"
    inputs = tokenizer(prefixed_input, return_tensors='pt', truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=150, min_new_tokens=20, num_beams=4, no_repeat_ngram_size=3, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Main Chatbot
def mental_health_chatbot(user_input):
    emotion = detect_emotion(user_input)
    response = generate_response(user_input, emotion)
    print(f"Detected Emotion: {emotion}")
    print(f"Bot: {response}")
    return {'emotion': emotion, 'response': response}

print("Pipeline Ready!")

In [ ]:
def mental_health_chatbot(user_input):
    emotion = detect_emotion(user_input)
    response = generate_response(user_input, emotion)
    print(f"User: {user_input}")
    print(f"Emotion: {emotion}")
    print(f"Bot: {response}")
    print("-" * 50)
    return {'emotion': emotion, 'response': response}

test_inputs = [
    "I lost my job today and I feel devastated",
    "I feel so anxious and cannot sleep at night",
    "I am angry at everything around me",
    "I feel hopeless and do not know what to do",
    "I feel so alone and nobody understands me",
    "I am scared about my future",
    "Everything makes me sad these days",
    "I am grateful for the support I received"
]

for user_input in test_inputs:
    mental_health_chatbot(user_input)

In [ ]:
def chat():
    print("MENTAL HEALTH CHATBOT")
    print("=" * 40)
    print("Type quit to exit\n")

    while True:
        user_input = input("You: ")
        if user_input.lower() == 'quit':
            print("Bot: Take care! Goodbye!")
            break
        if not user_input.strip():
            continue

        emotion = detect_emotion(user_input)
        response = generate_response(user_input, emotion)
        print(f"Emotion: {emotion}")
        print(f"Bot: {response}")
        print("-" * 40)

chat()